# SQL Joins — Runnable Examples

This notebook uses SQLite and pandas. All sample data is fictional. Run cells from top to bottom.

In [1]:
import sqlite3
import pandas as pd

connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
cursor = connection.cursor()
print("SQLite connection ready")

SQLite connection ready


## 1. Create sample tables

In [2]:
cursor.executescript('''
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    city TEXT NOT NULL
);
INSERT INTO customers VALUES
(1, 'Asha', 'Indore'),
(2, 'Ravi', 'Pune'),
(3, 'Meera', 'Indore'),
(4, 'Kabir', 'Mumbai'),
(5, 'Neha', 'Pune');

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT NOT NULL,
    amount REAL NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
INSERT INTO orders VALUES
(101, 1, '2026-01-10', 250, 'completed'),
(102, 1, '2026-01-12', 125, 'completed'),
(103, 2, '2026-01-15', 600, 'cancelled'),
(104, 3, '2026-02-01', 80, 'pending'),
(105, 4, '2026-02-03', 320, 'completed'),
(106, 2, '2026-02-08', 180, 'completed');

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL
);
INSERT INTO products VALUES
(1, 'Laptop', 'Electronics'),
(2, 'Notebook', 'Stationery'),
(3, 'Water Bottle', 'Accessories');
''')
print("customers, orders, and products tables created")

customers, orders, and products tables created


## 2. Inspect the tables

In [3]:
pd.read_sql_query("SELECT * FROM customers ORDER BY customer_id", connection)

,customer_id,customer_name,city
0,1,Asha,Indore
1,2,Ravi,Pune
2,3,Meera,Indore
3,4,Kabir,Mumbai
4,5,Neha,Pune


In [4]:
pd.read_sql_query("SELECT * FROM orders ORDER BY order_id", connection)

,order_id,customer_id,order_date,amount,status
0,101,1,2026-01-10,250.0,completed
1,102,1,2026-01-12,125.0,completed
2,103,2,2026-01-15,600.0,cancelled
3,104,3,2026-02-01,80.0,pending
4,105,4,2026-02-03,320.0,completed
5,106,2,2026-02-08,180.0,completed


## 3. INNER JOIN

In [5]:
pd.read_sql_query("""SELECT c.customer_name, o.order_id, o.amount, o.status
FROM customers AS c JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY o.order_id;""", connection)

,customer_name,order_id,amount,status
0,Asha,101,250.0,completed
1,Asha,102,125.0,completed
2,Ravi,103,600.0,cancelled
3,Meera,104,80.0,pending
4,Kabir,105,320.0,completed
5,Ravi,106,180.0,completed


## 4. LEFT JOIN — preserve customers without orders

In [6]:
pd.read_sql_query("""SELECT c.customer_name, o.order_id, o.amount
FROM customers AS c LEFT JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id;""", connection)

,customer_name,order_id,amount
0,Asha,101.0,250.0
1,Asha,102.0,125.0
2,Ravi,103.0,600.0
3,Ravi,106.0,180.0
4,Meera,104.0,80.0
5,Kabir,105.0,320.0
6,Neha,NaN,NaN


## 5. Find customers with no orders

In [7]:
pd.read_sql_query("""SELECT c.customer_id, c.customer_name
FROM customers AS c LEFT JOIN orders AS o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;""", connection)

,customer_id,customer_name
0,5,Neha


## 6. Filter in ON to preserve unmatched left rows

In [8]:
pd.read_sql_query("""SELECT c.customer_name, o.order_id, o.status
FROM customers AS c LEFT JOIN orders AS o
  ON c.customer_id = o.customer_id AND o.status = 'completed'
ORDER BY c.customer_id;""", connection)

,customer_name,order_id,status
0,Asha,101.0,completed
1,Asha,102.0,completed
2,Ravi,106.0,completed
3,Meera,NaN,None
4,Kabir,105.0,completed
5,Neha,NaN,None


## 7. Compare filter placement in WHERE

In [9]:
pd.read_sql_query("""SELECT c.customer_name, o.order_id, o.status
FROM customers AS c LEFT JOIN orders AS o ON c.customer_id = o.customer_id
WHERE o.status = 'completed' ORDER BY c.customer_id;""", connection)

,customer_name,order_id,status
0,Asha,101,completed
1,Asha,102,completed
2,Ravi,106,completed
3,Kabir,105,completed


## 8. Aggregate after a LEFT JOIN

In [10]:
pd.read_sql_query("""SELECT c.customer_name, COUNT(o.order_id) AS order_count,
COALESCE(SUM(o.amount), 0) AS total_amount
FROM customers AS c LEFT JOIN orders AS o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name ORDER BY c.customer_id;""", connection)

,customer_name,order_count,total_amount
0,Asha,2,375.0
1,Ravi,2,780.0
2,Meera,1,80.0
3,Kabir,1,320.0
4,Neha,0,0.0


## 9. HAVING after a join

In [11]:
pd.read_sql_query("""SELECT c.customer_name, COUNT(o.order_id) AS order_count
FROM customers AS c JOIN orders AS o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(o.order_id) > 1;""", connection)

,customer_name,order_count
0,Asha,2
1,Ravi,2


## 10. CROSS JOIN

In [12]:
pd.read_sql_query("""SELECT c.customer_name, p.product_name
FROM customers AS c CROSS JOIN products AS p
ORDER BY c.customer_id, p.product_id;""", connection)

,customer_name,product_name
0,Asha,Laptop
1,Asha,Notebook
2,Asha,Water Bottle
3,Ravi,Laptop
4,Ravi,Notebook
5,Ravi,Water Bottle
6,Meera,Laptop
7,Meera,Notebook
8,Meera,Water Bottle
9,Kabir,Laptop


## 11. Self join — customers from the same city

In [13]:
pd.read_sql_query("""SELECT c1.customer_name AS customer_1,
c2.customer_name AS customer_2, c1.city
FROM customers AS c1 JOIN customers AS c2
ON c1.city = c2.city AND c1.customer_id < c2.customer_id
ORDER BY c1.city;""", connection)

,customer_1,customer_2,city
0,Asha,Meera,Indore
1,Ravi,Neha,Pune


## 12. Join and aggregate by city

In [14]:
pd.read_sql_query("""SELECT c.city, COALESCE(SUM(o.amount), 0) AS total_order_amount
FROM customers AS c LEFT JOIN orders AS o ON c.customer_id = o.customer_id
GROUP BY c.city ORDER BY total_order_amount DESC;""", connection)

,city,total_order_amount
0,Pune,780.0
1,Indore,455.0
2,Mumbai,320.0


## Key takeaways

- `INNER JOIN` returns matches; `LEFT JOIN` retains every left-side row.
- `CROSS JOIN` creates every possible row combination.
- `ON` defines matching; `WHERE` filters the joined result.
- Check keys and relationship cardinality to avoid accidental row multiplication.
- For zero-count groups after a left join, use `COUNT(right_table.primary_key)`, not `COUNT(*)`.